In [161]:
## read in datasets and load libraries
import pandas as pd
import numpy as np

game_data = pd.read_csv('/Users/maryellenfaulconer/Documents/Basketball_Project/games_data.csv')
team_data = pd.read_csv('/Users/maryellenfaulconer/Documents/Basketball_Project/March_Madness.csv')

In [162]:
game_data.head()

,game_id,date,home_team,home_team_first_half_score,home_team_second_half_score,away_team,away_team_first_half_score,away_team_second_half_score,status
0,401824809,2025-11-04T01:00Z,Houston Cougars,44.0,31.0,Lehigh Mountain Hawks,23.0,34.0,Final
1,401826885,2025-11-04T00:00Z,Arizona Wildcats,50.0,43.0,Florida Gators,46.0,41.0,Final
2,401812785,2025-11-04T00:00Z,UConn Huskies,37.0,42.0,New Haven Chargers,24.0,31.0,Final
3,401820577,2025-11-03T23:30Z,St. John's Red Storm,54.0,54.0,Quinnipiac Bobcats,34.0,40.0,Final
4,401826083,2025-11-04T01:30Z,Michigan Wolverines,69.0,52.0,Oakland Golden Grizzlies,38.0,40.0,Final


In [163]:
## drop columns we won't be using
game_data = game_data.drop(columns=['status'])

In [164]:
## long format data, one row per team per game
home_data = game_data[['game_id', 'date', 'home_team', 'home_team_first_half_score', 'home_team_second_half_score']]
home_data = home_data.rename(columns={'home_team': 'team', 'home_team_first_half_score': 'first_half_score', 'home_team_second_half_score': 'second_half_score'})

## binary variable for home vs away (home = 1, away = 0)
home_data['home_away'] = 1
home_data['points_scored'] = home_data['first_half_score'] + home_data['second_half_score']
home_data['points_conceded'] = game_data['away_team_first_half_score'] + game_data['away_team_second_half_score']

## another binary variable for win vs loss (win = 1, loss = 0)
home_data['win_loss'] = (home_data['points_scored'] > home_data['points_conceded']).astype(int)

In [165]:
## same format for away teams
away_data = game_data[['game_id', 'date', 'away_team', 'away_team_first_half_score', 'away_team_second_half_score']]
away_data = away_data.rename(columns={'away_team': 'team', 'away_team_first_half_score': 'first_half_score', 'away_team_second_half_score': 'second_half_score'})

## binary variable for home vs away (home = 1, away = 0)
away_data['home_away'] = 0
away_data['points_scored'] = away_data['first_half_score'] + away_data['second_half_score']
away_data['points_conceded'] = game_data['home_team_first_half_score'] + game_data['home_team_second_half_score']

## another binary variable for win vs loss (win = 1, loss = 0)
away_data['win_loss'] = (away_data['points_scored'] > away_data['points_conceded']).astype(int)

In [166]:
## check number of rows in each dataset (should be the same)
print("Home data rows:", home_data.shape[0])
print("Away data rows:", away_data.shape[0])

Home data rows: 6195
Away data rows: 6195


In [167]:
## stack home and away data together
full_data = pd.concat([home_data, away_data], ignore_index=True)

In [168]:
## calculate avg + std dev points scored and conceded for each team, only using data progressively (i.e. only using games that occurred before the current game)
full_data = full_data.sort_values(by='date')

full_data['avg_points_scored'] = (
    full_data.groupby('team')['points_scored']
    .transform(lambda x: x.expanding().mean().shift())
)

full_data['std_points_scored'] = (
    full_data.groupby('team')['points_scored']
    .transform(lambda x: x.expanding().std().shift())
)

full_data['avg_points_conceded'] = (
    full_data.groupby('team')['points_conceded']
    .transform(lambda x: x.expanding().mean().shift())
)

full_data['std_points_conceded'] = (
    full_data.groupby('team')['points_conceded']
    .transform(lambda x: x.expanding().std().shift())
)

In [169]:
## show me one teams data to check
full_data[full_data['team'] == 'Duke Blue Devils'].head(10)

,game_id,date,team,first_half_score,second_half_score,home_away,points_scored,points_conceded,win_loss,avg_points_scored,std_points_scored,avg_points_conceded,std_points_conceded
170,401817228,2025-11-05T01:45Z,Duke Blue Devils,32.0,43.0,1,75.0,60.0,1,NaN,NaN,NaN,NaN
362,401817229,2025-11-08T18:30Z,Duke Blue Devils,45.0,50.0,1,95.0,54.0,1,75.000000,NaN,60.000000,NaN
6697,401817230,2025-11-12T00:00Z,Duke Blue Devils,49.0,65.0,0,114.0,59.0,1,85.000000,14.142136,57.000000,4.242641
664,401817231,2025-11-15T00:00Z,Duke Blue Devils,51.0,49.0,1,100.0,62.0,1,94.666667,19.502137,57.666667,3.214550
864,401817232,2025-11-19T02:00Z,Duke Blue Devils,41.0,37.0,1,78.0,66.0,1,96.000000,16.145175,58.750000,3.403430
1021,401813377,2025-11-22T00:00Z,Duke Blue Devils,47.0,53.0,1,100.0,42.0,1,92.400000,16.133815,60.200000,4.381780
1139,401817233,2025-11-23T21:00Z,Duke Blue Devils,52.0,41.0,1,93.0,56.0,1,93.666667,14.760307,57.166667,8.400397
7588,401817234,2025-11-28T01:00Z,Duke Blue Devils,41.0,39.0,0,80.0,71.0,1,93.571429,13.476611,57.000000,7.681146
1590,401806364,2025-12-03T00:30Z,Duke Blue Devils,36.0,31.0,1,67.0,66.0,1,91.875000,13.367738,58.750000,8.664377
7955,401817235,2025-12-06T17:00Z,Duke Blue Devils,31.0,35.0,0,66.0,60.0,1,89.111111,15.003703,59.555556,8.457410


In [170]:
## shape of full data
print("Full data shape:", full_data.shape)

Full data shape: (12390, 13)


In [171]:
## view team data
## number of unique teams in team data
summary = team_data['Team Name'].nunique()
print("Number of unique teams in team data:", summary)


## number of unique teams in full data
summary_full = full_data['team'].nunique()
print("Number of unique teams in full data:", summary_full)


Number of unique teams in team data: 69
Number of unique teams in full data: 727


In [172]:
## merge in team data for kenpom info
full_data = full_data.merge(team_data, left_on=['team'], right_on=['Team Name'], how='left')

In [173]:
## check merged data
## number of unique teams in merged data
summary_merged = full_data['team'].nunique()
print("Number of unique teams in merged data:", summary_merged)


Number of unique teams in merged data: 727


In [174]:
full_data.head()

,game_id,date,team,first_half_score,second_half_score,home_away,points_scored,points_conceded,win_loss,avg_points_scored,std_points_scored,avg_points_conceded,std_points_conceded,Team Name,Team,NetRtg,ORtg,DRtg,Luck
0,401823449,2025-11-03T13:00Z,Winthrop Eagles,40.0,41.0,1,81.0,74.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,401823449,2025-11-03T13:00Z,Queens University Royals,35.0,39.0,0,74.0,81.0,0,NaN,NaN,NaN,NaN,Queens University Royals,181.0,-1.44,115.8,117.2,0.067
2,401817194,2025-11-03T16:00Z,Bradley Braves,26.0,37.0,0,63.0,69.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,401817194,2025-11-03T16:00Z,St. Bonaventure Bonnies,34.0,35.0,1,69.0,63.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,401830655,2025-11-03T16:30Z,East Texas A&M Lions,68.0,51.0,1,119.0,60.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [175]:
full_data.drop(columns=['Team Name'], inplace=True)

In [176]:
## make a variable to indicate whether or not to use kenpom data, replace NAs with 0
## rename 'Team' column to 'kenpom_rank' to avoid confusion with 'team' column
full_data = full_data.rename(columns={'Team': 'kenpom_rank', "Luck ": 'Luck'})
full_data['kenpom_available'] = np.where(full_data['kenpom_rank'].isna(), 0, 1)

## replace NAs in kenpom_rank with a high number (e.g. 999) to indicate that the team is not ranked
full_data['kenpom_rank'] = full_data['kenpom_rank'].fillna(0)
full_data['NetRtg'] = full_data['NetRtg'].fillna(0)
full_data['ORtg'] = full_data['ORtg'].fillna(0)
full_data['DRtg'] = full_data['DRtg'].fillna(0)
full_data['Luck'] = full_data['Luck'].fillna(0)


In [177]:
## fill NANs from early games by assuming every team is "average" (i.e. using the mean of the available kenpom data for each variable)
league_avg_pts = full_data["points_scored"].mean()
league_std_pts = full_data["points_scored"].std()

full_data["avg_points_scored"] = full_data["avg_points_scored"].fillna(league_avg_pts)
full_data["std_points_scored"] = full_data["std_points_scored"].fillna(league_std_pts)

full_data["avg_points_conceded"] = full_data["avg_points_conceded"].fillna(league_avg_pts)
full_data["std_points_conceded"] = full_data["std_points_conceded"].fillna(league_std_pts)

In [178]:
## create kenpom differences for each match up. merge again based on game id to get the kenpom rank of the opposing team, then calculate the difference between the two teams' kenpom ranks for each game
full_data['kenpom_rank_diff'] = full_data['kenpom_rank'] - full_data.groupby('game_id')['kenpom_rank'].transform('max')
full_data['NetRtg_diff'] = full_data['NetRtg'] - full_data.groupby('game_id')['NetRtg'].transform('max')
full_data['ORtg_diff'] = full_data['ORtg'] - full_data.groupby('game_id')['ORtg'].transform('max')
full_data['DRtg_diff'] = full_data['DRtg'] - full_data.groupby('game_id')['DRtg'].transform('max')
full_data['Luck_diff'] = full_data['Luck'] - full_data.groupby('game_id')['Luck'].transform('max')
full_data['avg_points_scored_diff'] = full_data['avg_points_scored'] - full_data.groupby('game_id')['avg_points_scored'].transform('max')
full_data['std_points_scored_diff'] = full_data['std_points_scored'] - full_data.groupby('game_id')['std_points_scored'].transform('max')
full_data['avg_points_conceded_diff'] = full_data['avg_points_conceded'] - full_data.groupby('game_id')['avg_points_conceded'].transform('max')
full_data['std_points_conceded_diff'] = full_data['std_points_conceded'] - full_data.groupby('game_id')['std_points_conceded'].transform('max')

In [179]:
full_data.head()

,game_id,date,team,first_half_score,second_half_score,home_away,points_scored,points_conceded,win_loss,avg_points_scored,...,kenpom_available,kenpom_rank_diff,NetRtg_diff,ORtg_diff,DRtg_diff,Luck_diff,avg_points_scored_diff,std_points_scored_diff,avg_points_conceded_diff,std_points_conceded_diff
0,401823449,2025-11-03T13:00Z,Winthrop Eagles,40.0,41.0,1,81.0,74.0,1,74.416142,...,0,-181.0,0.00,-115.8,-117.2,-0.067,0.0,0.0,0.0,0.0
1,401823449,2025-11-03T13:00Z,Queens University Royals,35.0,39.0,0,74.0,81.0,0,74.416142,...,1,0.0,-1.44,0.0,0.0,0.000,0.0,0.0,0.0,0.0
2,401817194,2025-11-03T16:00Z,Bradley Braves,26.0,37.0,0,63.0,69.0,0,74.416142,...,0,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.0
3,401817194,2025-11-03T16:00Z,St. Bonaventure Bonnies,34.0,35.0,1,69.0,63.0,1,74.416142,...,0,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.0
4,401830655,2025-11-03T16:30Z,East Texas A&M Lions,68.0,51.0,1,119.0,60.0,1,74.416142,...,0,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.0


In [180]:
## save data frame as fully completed dataset for modeling
full_data.to_csv('/Users/maryellenfaulconer/Documents/Basketball_Project/full_data.csv', index=False)